# Demo 4 · Validating and tuning models

In Demo 1, we judged a model with **one** train/validation split. This notebook asks two
questions that come up in every machine-learning project:

1. *How good is my model, really?* One split can be lucky or unlucky.
2. *Which settings should I use?* Most models have **hyperparameters** (values we
   choose before training, such as the number of neighbours in k-NN) and we need a fair
   way to pick them.

This demo is meant to be read at your own pace. Run each cell, read the output, and try
the **Your turn** exercise before opening its reference answer.

By the end, you will be able to:

- Explain why K-fold cross-validation gives a more reliable estimate than a single split, and when to use the stratified version.
- Use `cross_val_score` to compare models.
- Tune hyperparameters with `GridSearchCV` and `RandomizedSearchCV`, and read their results.
- Chain preprocessing and a model in a `Pipeline` so that tuning never sees the test data.

**The rule we follow throughout:** set a test set aside → choose and tune the model with
cross-validation on the training set only → evaluate **once** on the test set.

This demo has three notebooks: **01 (this one)**, then
[02 · Bootstrap confidence intervals](02_bootstrap_confidence_intervals.ipynb), then
[03 · Titanic, end to end](03_titanic_end_to_end.ipynb).

## Setup: choose local or Colab

**Locally:** run `uv sync` from the `ift3700-6758` repository root, then select the shared Python 3.11 `.venv` kernel. See the [repository README](../../../README_en.md) for setup. Skip the optional installation cell: it installs nothing locally.

**Google Colab | no repository clone or manual uv installation needed:**

1. Open this notebook in Colab and choose a **CPU** runtime. Installation needs internet access.
2. In the **optional cell below**, set `INSTALL_COLAB_PACKAGES = True` and run it once. The cell installs `uv`, then uses it to install the libraries into the notebook's current Python.
3. If Colab requests a restart after installation, choose **Runtime → Restart session**.
4. Set the flag back to `False`, then run the cells from top to bottom with **Shift+Enter**.

This notebook uses **NumPy** and **pandas** for arrays and tables, **matplotlib** for plots, **scikit-learn** for the models and **SciPy** for random-search distributions. The data (handwritten digits) is bundled with scikit-learn, so no download is needed.

In [ ]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = False  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "numpy>=1.26",
        "pandas>=2.2,<4",
        "matplotlib>=3.7",
        "scikit-learn>=1.4",
        "scipy>=1.11",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Restart the session if Colab requests it.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import loguniform

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## A. The data: handwritten digits

Each image is 8×8 pixels with grey levels from 0 to 16. scikit-learn expects a table with
one row per example and one column per feature, so each image is flattened into
**64 features**. The target `y` is the digit (0 to 9).

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
print("X shape:", X.shape, "| y shape:", y.shape)

fig, axes = plt.subplots(1, 8, figsize=(10, 1.8))
for ax, image, label in zip(axes, digits.images, digits.target):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(label)
    ax.axis("off")
plt.show()

## B. Why one split is not enough

Let us train the same decision tree five times, changing only **which rows** go to the
validation set (`random_state`). The model and data are identical; only the split changes.

In [ ]:
for seed in range(5):
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.3, random_state=seed)
    tree = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
    print(f"split {seed}: validation accuracy = {tree.score(X_val, y_val):.3f}")

The accuracy moves by about two percentage points from one split to the next. If two
models differed by that much on a single split, we could not tell whether one is really
better or just got a luckier split. **Cross-validation** averages
over several splits instead.

## C. K-fold cross-validation

K-fold cross-validation cuts the training data into *K* parts (**folds**). Each fold takes
one turn as the validation set while the model trains on the other *K − 1* folds. We get
*K* scores and report their mean and spread.

![5-fold cross-validation (scikit-learn documentation)](https://scikit-learn.org/stable/_images/grid_search_cross_validation.png)

The diagram also shows the rule from the introduction: the **test data** stays outside the
loop and is used only for the final evaluation.

### A tiny example to see the folds
Ten examples with three unbalanced classes: two `a`, two `b`, six `c`. The rows are
**sorted by class**, which happens often in real files.

In [ ]:
X_toy = np.arange(10).reshape(-1, 1)
y_toy = np.array(["a", "a", "b", "b", "c", "c", "c", "c", "c", "c"])

k_fold = KFold(n_splits=2, shuffle=False)
for fold, (train_idx, test_idx) in enumerate(k_fold.split(X_toy, y_toy), start=1):
    print(f"fold {fold}: train labels {y_toy[train_idx]} | validation labels {y_toy[test_idx]}")

In fold 1 the model would train on **only `c`** and be asked to recognise `a` and `b`, which
it has never seen. Plain K-fold does not look at the labels.

### Stratified K-fold
`StratifiedKFold` builds folds that keep the **class proportions** of the full dataset.
`shuffle=True` also mixes the rows first; `random_state` makes the shuffle reproducible.

In [ ]:
stratified = StratifiedKFold(n_splits=2, shuffle=True, random_state=0)
for fold, (train_idx, test_idx) in enumerate(stratified.split(X_toy, y_toy), start=1):
    print(f"fold {fold}: train labels {y_toy[train_idx]} | validation labels {y_toy[test_idx]}")

Each fold now contains every class in roughly the right proportion. For classification,
stratified folds are the usual choice.

### `cross_val_score`: cross-validation in one line
`cross_val_score` runs the whole loop for us: split, train, score, repeat. When you pass a
classifier and an integer `cv`, scikit-learn uses stratified folds automatically. We pass a
`StratifiedKFold` object explicitly so that the shuffle is reproducible.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=cv)
print("fold scores:", np.round(scores, 3))
print(f"mean = {scores.mean():.3f}, standard deviation = {scores.std():.3f}")

The mean is a more stable estimate than any single split, and the standard deviation tells
us how much the score depends on which data the model saw.

### Your turn
Use `cross_val_score` with the same `cv` to compare `KNeighborsClassifier` for
`n_neighbors` in `[1, 3, 5, 9, 15]`. Print the mean and standard deviation for each value.
Which *k* would you choose?

In [ ]:
# Your turn: loop over the values of k and print the cross-validation scores.

<details>
<summary>Reference answer — open after trying</summary>

```python
for k in [1, 3, 5, 9, 15]:
    knn_scores = cross_val_score(KNeighborsClassifier(n_neighbors=k), X, y, cv=cv)
    print(f"k = {k:2d}: {knn_scores.mean():.3f} ± {knn_scores.std():.3f}")
```

All values of *k* score around 0.98–0.99, far above the decision tree. *k* = 1, 3 and 5
are within about one standard deviation of each other, while *k* = 15 starts to drop, so
any small *k* is a reasonable choice. Writing this loop by hand is exactly what
`GridSearchCV` automates in the next section.

</details>

## D. Hyperparameter search

A **support vector classifier** (`SVC`) has several hyperparameters:

- `kernel`: the shape of the decision boundary (`"linear"` or the more flexible `"rbf"`).
- `C`: how strongly to penalise training mistakes. A large `C` fits the training data more closely.
- `gamma` (rbf only): how far the influence of one training example reaches.

We first set aside **half of the data as a test set**. All tuning happens on the other half
(the *development* set) with cross-validation.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.5, stratify=y, random_state=0
)
print("development set:", X_dev.shape, "| test set:", X_test.shape)

### (a) Grid search
`GridSearchCV` tries **every combination** in a grid. For each combination it runs
cross-validation on the development set, then refits the best combination on the whole
development set.

Here the grid has 2 × 4 = 8 rbf combinations plus 4 linear ones: 12 candidates × 5 folds =
60 fits. We score with **macro F1** (the average F1 over the ten classes), which treats every
digit equally.

In [ ]:
param_grid = [
    {"kernel": ["rbf"], "gamma": [1e-3, 1e-4], "C": [1, 10, 100, 1000]},
    {"kernel": ["linear"], "C": [1, 10, 100, 1000]},
]

grid_search = GridSearchCV(SVC(), param_grid, scoring="f1_macro", cv=cv)
grid_search.fit(X_dev, y_dev)

print("best parameters:", grid_search.best_params_)
print(f"best cross-validated macro F1: {grid_search.best_score_:.3f}")

`cv_results_` keeps the score of every candidate. A table sorted by rank shows whether the
winner is clearly better or only marginally so.

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
columns = ["param_kernel", "param_C", "param_gamma", "mean_test_score", "std_test_score", "rank_test_score"]
display(results[columns].sort_values("rank_test_score").round(4))

The top three candidates are tied: with `gamma = 0.001`, any `C` of 10 or more gives the
same folds the same predictions. Many candidates are within one standard deviation of the
best, so on this data the exact choice matters little.

Now, and only now, we evaluate the selected model **once** on the test set. The report
gives precision, recall and F1 for each digit.

In [ ]:
y_pred = grid_search.predict(X_test)
print(classification_report(y_test, y_pred))

### (b) Randomized search
A grid grows quickly: 4 hyperparameters with 10 values each already means 10,000
candidates. `RandomizedSearchCV` instead **samples** a fixed number of candidates
(`n_iter`) from distributions that we describe.

`loguniform(1, 1000)` draws values spread evenly on a **log scale** between 1 and 1000,
which suits `C` and `gamma` because what matters is their order of magnitude. With the
same budget of 12 candidates, random search explores many more distinct values of each
hyperparameter than the grid did.

In [ ]:
param_distributions = [
    {"kernel": ["rbf"], "gamma": loguniform(1e-4, 1e-3), "C": loguniform(1, 1000)},
    {"kernel": ["linear"], "C": loguniform(1, 1000)},
]

random_search = RandomizedSearchCV(
    SVC(), param_distributions, n_iter=12, scoring="f1_macro", cv=cv, random_state=0
)
random_search.fit(X_dev, y_dev)

print("best parameters:", random_search.best_params_)
print(f"best cross-validated macro F1: {random_search.best_score_:.3f}")
print(f"test macro F1: {f1_score(y_test, random_search.predict(X_test), average='macro'):.3f}")

Both searches reach about the same score here because SVC is not very sensitive to these
values on digits. Random search pays off when there are many hyperparameters and only a few
of them really matter.

## E. Pipelines

Real models usually need **preprocessing** before the estimator: scaling, dimensionality
reduction, encoding categories… A `Pipeline` chains these steps into a single object with
`fit` and `predict`. This brings three benefits:

- **Convenience:** one `fit` and one `predict` call run every step in order.
- **Joint tuning:** a grid search can tune the preprocessing and the model together.
- **No leakage:** preprocessing steps also learn from data. `PCA` learns a mean and
  directions; `StandardScaler` learns means and standard deviations. If we fit them on the full dataset
  before cross-validation, the validation folds have already influenced the model and the
  scores are too optimistic. Inside a pipeline, each fold refits the preprocessing on its
  own training part only.

Our pipeline: keep the first *n* PCA components of the pixels → logistic regression. Pipeline parameters are named `step__parameter`, with **two** underscores.

In [ ]:
pipe = Pipeline(steps=[
    ("pca", PCA()),
    # A large tolerance keeps this example fast.
    ("logistic", LogisticRegression(max_iter=10_000, tol=0.1)),
])

param_grid = {
    "pca__n_components": [5, 15, 30, 45, 64],
    "logistic__C": np.logspace(-4, 4, 4),
}

search = GridSearchCV(pipe, param_grid, cv=cv)
search.fit(X_dev, y_dev)

print("best parameters:", search.best_params_)
print(f"best cross-validated accuracy: {search.best_score_:.3f}")

In [ ]:
print(classification_report(y_test, search.predict(X_test)))

This pipeline is weaker than the tuned SVC above (about 0.93 against 0.98): logistic
regression draws straight boundaries. The point of this section is the mechanics, which
are the same for any model.

### How many components did we keep?
The plot shows the **cumulative explained variance**: the share of the pixels' variation that
the first *n* components preserve. The dotted line marks the number chosen by the search.

In [ ]:
best_pca = search.best_estimator_.named_steps["pca"]
pca_full = PCA().fit(X_dev)

plt.plot(np.arange(1, pca_full.n_components_ + 1), np.cumsum(pca_full.explained_variance_ratio_))
plt.axvline(best_pca.n_components, linestyle=":", color="black", label="n_components chosen")
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance")
plt.legend()
plt.show()

The search did not simply keep as much variance as possible: it picked the number of
components that gave the best **validation** score.

## Takeaways

- A single split gives a noisy estimate. **K-fold cross-validation** averages over *K* splits; use **stratified** folds for classification.
- **Grid search** tries every combination; **randomized search** samples a fixed budget and scales better to many hyperparameters.
- Tune on the development set with cross-validation, then evaluate **once** on the test set.
- Put preprocessing inside a **pipeline** so that every fold learns it from training data only.

**Next:** [02 · Bootstrap confidence intervals](02_bootstrap_confidence_intervals.ipynb)
asks how uncertain a score is.

**To go further:** scikit-learn user guide on
[cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html),
[hyperparameter search](https://scikit-learn.org/stable/modules/grid_search.html) and
[pipelines](https://scikit-learn.org/stable/modules/compose.html).